# ==============================================================================
# 1. DESEQ + ATAC Consensus peak calling
# ==============================================================================

In [1]:
"""
End-to-end pipeline: CD133 E14 vs E18
DESeq2 -> Consensus Peaks -> Enhancer Integration -> CellOracle Base GRN -> GRN Pruning
"""

import os
import pandas as pd
import numpy as np
from importlib import reload
import pandas as pd 
import numpy as np 
import sys
import os
# get project root (two levels up from this notebook)
project_root = os.path.abspath(os.path.join(os.path.dirname('src'), '..'))
# if in notebook:
# project_root = os.path.abspath('..')   # or adjust as needed
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.pipeline as pipeline
import src.enhancer_atac as enatac
import src.grn_pruner as grnpruner
# Reload modules
reload(pipeline)
reload(enatac)
reload(grnpruner)

# ==============================================================================
# PATHS
# ==============================================================================

base_dir = "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC"

paths = {
    # RNA-seq
    "counts": os.path.join(base_dir, "data/rna_seq/E14_ E18_LGE_cortex_seq_Counts.csv"),
    "metadata": os.path.join(base_dir, "data/rna_seq/Samples+Pooling_RNA-Seq.csv"),
    
    # ATAC-seq
    "atac_metadata": os.path.join(base_dir, "data/atac_seq/ATAC-seq/samples_clean.csv"),
    "atac_peak_dir": os.path.join(base_dir, "data/atac_seq/ATAC-seq"),
    
    # Annotations
    "tf_list": os.path.join(base_dir, "data/annotations/Mouse_TFs_Kinases_webpage-3-30-2017.xlsx"),
    "gtf": os.path.join(base_dir, "data/annotations/gencode.vM36.annotation.gtf"),
    "enhancer_files": [
        os.path.join(base_dir, "data/annotations/enhancerAtlas_neuron_cortical.txt"),
        # os.path.join(base_dir, "data/annotations/enhancerAtlas_brain_e14.5.txt"),
        os.path.join(base_dir, "data/annotations/enhancerAtlas_cortex.txt")
    ],
    
    # Integrated data (for pipeline object only)
    "overlap_df": os.path.join(base_dir, "data/integrated/overlap_annotated.tsv"),
    "chip_annotated": os.path.join(base_dir, "data/integrated/chip_annotated_filtered.tsv"),
    "atac_annotated": os.path.join(base_dir, "data/integrated/atac_annotated.tsv"),
    
    # Output paths
    "output_dir": os.path.join(base_dir, "results/cd133_e14_e18_grn"),
    "consensus_bed": os.path.join(base_dir, "results/cd133_e14_e18_grn/consensus_cd133.bed"),
    "deseq_output": os.path.join(base_dir, "results/cd133_e14_e18_grn/deseq_temporal_cd133.csv"),
    "celloracle_h5": os.path.join(base_dir, "results/cd133_e14_e18_grn/celloracle_tfinfo.h5"),
    "celloracle_parquet": os.path.join(base_dir, "results/cd133_e14_e18_grn/celloracle_base_grn.parquet"),
    "e14_grn": os.path.join(base_dir, "results/cd133_e14_e18_grn/E14_TF_network.csv"),
    "e18_grn": os.path.join(base_dir, "results/cd133_e14_e18_grn/E18_TF_network.csv")
}

# Create output directory
os.makedirs(paths["output_dir"], exist_ok=True)

exclude_samples = ['MUC9939', 'MUC9940', 'MUC9914']

print("="*80)
print("CD133 E14 vs E18 GRN PIPELINE")
print("="*80)

# ==============================================================================
# STEP 1: RNA-SEQ DIFFERENTIAL EXPRESSION (CD133 E14 vs E18)
# ==============================================================================

print("\n" + "="*80)
print("STEP 1: DIFFERENTIAL EXPRESSION ANALYSIS")
print("="*80)

nsc = pipeline.NSCAnalysis(
    counts_path=paths['counts'],
    metadata_path=paths['metadata'],
    atac_metadata_path=paths['atac_metadata'],
    overlap_df_path=paths['overlap_df'],
    chip_annotated_path=paths['chip_annotated'],
    atac_annotated_path=paths['atac_annotated'],
    tf_list_path=paths['tf_list'],
    gtf_path=paths['gtf'],
    exclude_samples=exclude_samples
)

# Filter for CD133 only
cd133_samples = nsc.metadata[nsc.metadata['Marker'] == 'Progenitors'].index.tolist()
nsc.counts = nsc.counts[[c for c in cd133_samples if c in nsc.counts.columns]]
nsc.metadata = nsc.metadata.loc[cd133_samples]

print(f"\nCD133 samples: {len(nsc.metadata)}")
print(nsc.metadata[['Stage', 'Region', 'Marker']].value_counts())

# Run DESeq2
deseq_results = nsc.run_deseq(group1='E14', group2='E18', group_col='Stage')

# Save DESeq results
deseq_results.to_csv(paths['deseq_output'])
print(f"\nDESeq results saved to: {paths['deseq_output']}")

# Get DE TFs
de_tfs = deseq_results[
    (deseq_results['padj'] < 0.05) & 
    (deseq_results['log2FoldChange'].abs() > 1) &
    (deseq_results['is_TF'] == True)
]

deg_list_e14 = de_tfs[de_tfs['log2FoldChange'] > 1]['symbol'].tolist()
deg_list_e18 = de_tfs[de_tfs['log2FoldChange'] < -1]['symbol'].tolist()

print(f"\nE14-high TFs: {len(deg_list_e14)}")
print(f"E18-high TFs: {len(deg_list_e18)}")
print(f"\nE14 TFs: {deg_list_e14[:10]}...")
print(f"E18 TFs: {deg_list_e18[:10]}...")

# ==============================================================================
# STEP 2: BUILD CONSENSUS PEAKS FROM ATAC-SEQ
# ==============================================================================

print("\n" + "="*80)
print("STEP 2: CONSENSUS PEAK BUILDING")
print("="*80)

builder = enatac.ConsensusPeakBuilder(
    metadata_file=paths['atac_metadata'],
    peak_dir=paths['atac_peak_dir'],
    factor='CD133',
    conditions=['E14', 'E18']
)

consensus = builder.build_consensus()
builder.save_bed(consensus, paths['consensus_bed'])

print(f"\nConsensus peaks: {len(consensus)}")
print(f"Saved to: {paths['consensus_bed']}")

# ==============================================================================
# STEP 3: ENHANCER INTEGRATION
# ==============================================================================

print("\n" + "="*80)
print("STEP 3: ENHANCER ATLAS INTEGRATION")
print("="*80)

integrator = enatac.EnhancerIntegrator(
    enhancer_files=paths['enhancer_files'],
    from_assembly='mm9'
)

enhancers_mm39 = integrator.load_and_liftover()

# Overlap with consensus peaks
target_genes, overlaps_df = integrator.overlap_with_consensus(consensus)

# Get DE TFs with accessible enhancers
e14_tfs_with_enh = integrator.get_de_with_accessible_enhancers(target_genes, deg_list_e14)
e18_tfs_with_enh = integrator.get_de_with_accessible_enhancers(target_genes, deg_list_e18)

print(f"\nE14 TFs with accessible enhancers: {e14_tfs_with_enh}")
print(f"E18 TFs with accessible enhancers: {e18_tfs_with_enh}")

# Save enhancer overlaps
overlaps_df.to_csv(os.path.join(paths['output_dir'], 'enhancer_consensus_overlaps.csv'), index=False)


/home/users/adhal/micromamba/envs/scrna_target_idf/lib/python3.10/site-packages/sorted_nearest/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


CD133 E14 vs E18 GRN PIPELINE

STEP 1: DIFFERENTIAL EXPRESSION ANALYSIS
Loading data...

[1/9] Loading counts...
      Dropping 1 genes with NaN values
      28664 genes x 36 samples

[2/9] Loading RNA-seq metadata...
      36 samples
      Columns: ['ForeignID', 'Stage', 'Replicate', 'Region', 'Marker', 'Shuffle']

[3/9] Loading ATAC-seq metadata...
      15 samples
      Columns: ['Tissue', 'Factor', 'Condition', 'Treatment', 'Replicate', 'bamReads', 'Peaks', 'PeakCaller']

[4/9] Creating ATAC-to-RNA sample mapping...
      Mapped: 15 / 15 ATAC samples

[5/9] Removing outlier samples...
      Removed: ['MUC9939', 'MUC9940', 'MUC9914']
      Remaining: 33 samples

[6/9] Filtering low-count genes...

Filtering low-count genes
Threshold: counts >= 10 in >= 3 samples
  E14: 17483 genes pass filter
  E18: 17939 genes pass filter

Genes before: 28664
Genes after:  18187
Removed:      10477 (36.6%)

[7/9] Loading overlap_df (ChIP ∩ ATAC)...
      448802 overlaps
      Columns: ['chip_chr', 

Fitting size factors...
... done in 0.03 seconds.

Fitting dispersions...
... done in 26.16 seconds.

Fitting dispersion trend curve...
... done in 0.80 seconds.

Fitting MAP dispersions...
... done in 39.60 seconds.

Fitting LFCs...
... done in 13.50 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 14 outlier genes.

Fitting dispersions...
... done in 0.02 seconds.

Fitting MAP dispersions...
... done in 0.02 seconds.

Fitting LFCs...
... done in 0.01 seconds.

Running Wald tests...


Extracting results (E14 vs E18)...


... done in 4.40 seconds.



Log2 fold change & Wald test p-value: Stage E14 vs E18
                        baseMean  log2FoldChange     lfcSE      stat  \
ENSMUSG00000000001  11392.735011        0.163045  0.071729  2.273084   
ENSMUSG00000000088   7290.014219        0.374888  0.087763  4.271573   
ENSMUSG00000000581   2662.931320       -0.024108  0.077363 -0.311623   
ENSMUSG00000006471   2191.953433        0.199988  0.038724  5.164521   
ENSMUSG00000036019    465.437536       -0.415655  0.145974 -2.847467   
...                          ...             ...       ...       ...   
ENSMUSG00000035984    171.993688       -4.000665  0.435222 -9.192241   
ENSMUSG00000035992   1390.075479       -0.098378  0.124280 -0.791582   
ENSMUSG00000036002   2360.311704       -0.017796  0.059961 -0.296788   
ENSMUSG00000036006    606.543527       -1.622156  0.212070 -7.649142   
ENSMUSG00000036009    659.660379       -0.053090  0.080047 -0.663239   

                          pvalue          padj  
ENSMUSG00000000001  2.302114e-0

/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:657: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_sig_tf = results[sig_mask & results['is_TF']].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:658: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_sig_gene = results[sig_mask & ~results['is_TF']].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:660: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_up = results[sig_mask & (results['log2FoldChange'] > 0)].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:661: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_down = results[sig_mask & (results['log2FoldChange'] < 0)].shape[0]



DESeq results saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/deseq_temporal_cd133.csv

E14-high TFs: 39
E18-high TFs: 83

E14 TFs: ['Hmga2', 'Smad3', 'Rcor2', 'Plagl2', 'Sox3', 'Sall4', 'Lhx9', 'Prdm12', 'Lef1', 'Hmga1']...
E18 TFs: ['Csdc2', 'Creb3l2', 'Egr1', 'Etv4', 'Dbx2', 'Klf9', 'Nfatc1', 'Sox10', 'Foxo1', 'Zbtb7c']...

STEP 2: CONSENSUS PEAK BUILDING
Found 10 samples for CD133 in ['E14', 'E18']
Consensus peaks: 93492
Saved to /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/consensus_cd133.bed

Consensus peaks: 93492
Saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/consensus_cd133.bed

STEP 3: ENHANCER ATLAS INTEGRATION
Initializing liftOver from mm9 to mm39...
Loading enhancerAtlas_neuron_cortical.txt
Loading enhancerAtlas_cortex.txt
Loaded 124051 unique enhancer-gene pairs in mm9
Lifting over to mm39...
LiftOver: 124012/124051 (100.0%)


In [2]:
overlaps_df

,Chromosome,Start,End,symbol,score,Start_b,End_b
0,chr1,166082028,166088968,Ildr2,7.504045,166081475,166082121
1,chr1,166082028,166088968,Ildr2,7.504045,166083479,166083583
2,chr1,75358089,75358679,Gm15178,7.184706,75358306,75359324
3,chr1,168233508,168255428,Pbx1,6.908724,168238327,168238506
4,chr1,168233508,168255428,Pbx1,6.908724,168244539,168244936
...,...,...,...,...,...,...,...
87802,chrX,133196370,133198000,Taf7l,0.012776,133196049,133196561
87803,chrX,133196370,133198000,Taf7l,0.012776,133197558,133197705
87804,chrX,7764873,7765003,Gripap1,0.012589,7764995,7766007
87805,chrX,7765233,7765403,Gripap1,0.012589,7764995,7766007


# ==============================================================================
# 2. CELLORACLE BASE GRN CONSTRUCTION
# ==============================================================================

In [5]:
import src.base_grn as base_grn

In [6]:

import numpy as np
import pandas as pd
import seaborn as sns
import sys, os

import matplotlib.pyplot as plt
from gimmemotifs.motif import Motif,read_motifs

## 2.1. Custom motif maker

In [3]:


# Load TF information as a dataframe.
df = pd.read_table("/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations/TF_Information_all_motifs.txt")
df.head()
# All process will be done inside these function.

from datetime import datetime
import glob

def read_pwn_and_convert_into_list(path):
    # read pwn as df
    pwm = pd.read_csv(path, delimiter="\t")

    # convert into list of str
    li = []
    for i in pwm.iterrows():
        i = i[1].values[1:]
        i = "\t".join(i.astype("str")) + "\n"
        li.append(i)

    return li

def make_motif_file_from_cisbp_data(pwm_folder_path, tfinfo_df, species):

    data_ = tfinfo_df[tfinfo_df.TF_Species == species]
    data_name = "CisBP_ver2_" + species

    ## 1. Make file: motif2factors.txt

    # Select information
    df_factors = data_[["Motif_ID", "TF_Name", "MSource_Type", "TF_Status"]]
    df_factors = df_factors[df_factors.TF_Status != "N"]

    # Formatting
    df_factors.columns = 'Motif\tFactor\tEvidence\tCurated'.split("\t")
    df_factors["Curated"] = [{"D": "Y", "I": "N"}[i] for i in df_factors["Curated"]]
    df_factors = df_factors.sort_values(by="Motif")

    ## 2. Make file: pfm file
    comments = f"# CIS-BP motif database (v2.0), retrieved by Celloracle\n"
    comments += "# Retrieved from: http://cisbp.ccbr.utoronto.ca/data/2.00/DataFiles/Bulk_downloads/EntireDataset/PWMs.zip\n"
    comments += f"#Date: {datetime.now().ctime()}\n"

    # Get list of motif name
    paths_pwm = glob.glob(os.path.join(pwm_folder_path, "*.txt"))
    paths_pwm.sort()
    motif_names = [path.split("/")[-1].replace(".txt", "") for path in paths_pwm]
    motifs = np.intersect1d(motif_names, df_factors.Motif.unique())

    print(motifs.shape)

    # Intersect motif information with pwm information
    df_factors = df_factors[df_factors.Motif.isin(motifs)]

    # Load, convert, and save pwm info
    output = data_name + ".pfm"

    motifs_non_zero = []
    with open(output, "w") as f:

        for motif_name in motifs:

            path = os.path.join(pwm_folder_path, motif_name + ".txt")
            pwm = read_pwn_and_convert_into_list(path=path) # Load and convert
            if pwm:
                motifs_non_zero.append(motif_name)
                pwm = [f">{motif_name}\n"] + pwm
                for i in pwm: # Save pfm
                    f.write(i)


    # Intersect motif information with pwm information
    df_factors = df_factors[df_factors.Motif.isin(motifs_non_zero)]

    # Save factor info
    df_factors.to_csv(f'{data_name}.motif2factors.txt', sep='\t', index=False)

    print(df_factors.shape, len(motifs_non_zero))
species = 'Mus_musculus'
make_motif_file_from_cisbp_data(pwm_folder_path="/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations/pwms", tfinfo_df=df, species=species)

(4600,)
(10760, 4) 4149


In [11]:
from gimmemotifs.motif import read_motifs
path_pfm = '/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations/pfm/CisBP_ver2_Mus_musculus.pfm'
# Check path for pfm file
print(path_pfm)

# Read motifs
motifs = read_motifs(path_pfm)

# Check first 10 motifs
motifs[:10]

/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations/pfm/CisBP_ver2_Mus_musculus.pfm


[M00063_3.00_nnnAAww,
 M00099_3.00_nTGTTTAyn,
 M00111_3.00_TAATAAAT,
 M00115_3.00_nnnTTCnnn,
 M00166_3.00_sCCnnrGGCn,
 M00167_3.00_TnGCCysAGG,
 M00168_3.00_nsCCnnAGGs,
 M00169_3.00_nnGCCynnGG,
 M00170_3.00_nTTTnATnn,
 M00171_3.00_nnTAATATTnn]

## 2.2. Base GRN using CO

In [16]:
print("\n" + "="*80)
print("STEP 4: CELLORACLE BASE GRN CONSTRUCTION")
print("="*80)


# Initialize CellOracle
co = base_grn.GRNCo(
    bed_path=paths['consensus_bed'],
    ref_genome='mm39',
    genomes_dir=None
)

# Step 1: Load BED
print("\nLoading consensus peaks...")
co.load_bed()

# Step 2: Annotate TSS
print("Annotating TSS...")
co.annotate_tss()

# Step 3: Ensure genome
print("Checking genome installation...")
co.ensure_genome()


# Step 4: Scan motifs
print("Scanning TF motifs (this takes time)...")
co.scan_motifs(fpr=0.02, verbose=True, motifs=None)

# Step 5: Filter motifs
print("Filtering motifs...")
co.filter_motifs(score_threshold=10)



STEP 4: CELLORACLE BASE GRN CONSTRUCTION

Loading consensus peaks...
Annotating TSS...
que bed peaks: 93492
tss peaks in que: 36221
Checking genome installation...
Scanning TF motifs (this takes time)...
No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 



2025-12-17 11:04:55,284 - DEBUG - using background: genome mm39 with size 200


Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 



2025-12-17 11:04:58,956 - DEBUG - determining FPR-based threshold


Motif scan started .. It may take long time.



...

Filtering motifs...
Filtering finished: 12160990 -> 2047703
1. Converting scanned results into one-hot encoded dataframe.


  0%|          | 0/17435 [00:00<?, ?it/s]

2. Converting results into dictionaries.


  0%|          | 0/18608 [00:00<?, ?it/s]

  0%|          | 0/1139 [00:00<?, ?it/s]

## 2.3. Adding TSS distance to annotated peaks

In [17]:
def add_tss_distance_pyranges(base_grn_annotation, gtf_file):
    """Using PyRanges for genomic distance calculation."""
    import pandas as pd
    import numpy as np
    import pyranges as pr
    
    # 1. Parse peak coordinates
    base_grn_annotation = base_grn_annotation.copy()
    base_grn_annotation[['chr', 'start', 'end']] = base_grn_annotation['peak_id'].str.extract(
        r'(chr[^_]+)_(\d+)_(\d+)'
    )
    base_grn_annotation['start'] = base_grn_annotation['start'].astype(int)
    base_grn_annotation['end'] = base_grn_annotation['end'].astype(int)
    
    # 2. Load GTF with PyRanges
    gtf = pr.read_gtf(gtf_file)
    
    # Get transcripts only
    transcripts = gtf[gtf.Feature == "transcript"]
    
    # Calculate TSS based on strand
    tss_df = transcripts.df
    tss_df['tss'] = np.where(tss_df['Strand'] == '+', 
                              tss_df['Start'], 
                              tss_df['End'])
    
    # One TSS per gene
    tss_per_gene = tss_df.groupby('gene_name').agg({
        'Chromosome': 'first',
        'tss': 'first'
    }).reset_index()
    tss_per_gene.columns = ['gene_short_name', 'gene_chr', 'gene_tss']
    
    # 3. Merge
    merged = base_grn_annotation.merge(tss_per_gene, on='gene_short_name', how='left')
    
    # Calculate distance
    merged['peak_center'] = (merged['start'] + merged['end']) / 2
    merged['distance_to_tss'] = np.abs(merged['peak_center'] - merged['gene_tss'])
    merged['distance_to_tss'].fillna(1e6, inplace=True)
    
    return merged[['peak_id', 'gene_short_name', 'chr', 'start', 'end', 'distance_to_tss']]

# Usage:
peak_annot = add_tss_distance_pyranges(
    base_grn_annotation=co.tss_annotated,
    gtf_file=paths['gtf']
)


In [21]:
co.tss_annotated

,peak_id,gene_short_name
0,chr1_3740869_3742728,Xkr4
1,chr1_3584982_3585271,Gm37329
2,chr1_4855424_4856438,Mrpl15
3,chr1_4855424_4856438,Mrpl15
4,chr1_4855424_4856438,Mrpl15
...,...,...
36216,chrY_90795052_90796359,Gm47283
36217,chrY_90795052_90796359,Gm47283
36218,chrY_90795052_90796359,Gm47283
36219,chrY_90795052_90796359,Gm47283


# ==============================================================================
# 3. GRN PRUNING
# ==============================================================================

In [18]:
tf_df = pd.read_excel(paths['tf_list'])
tf_df = tf_df.dropna()
tf_df = tf_df.iloc[1:, :]
tf_df.columns  = ['Symbol', 'Annotation', 'Family', 'Ensembl_ID', 'uniprot_ID']

In [19]:
paths['celloracle_h5'] = '/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/cd133.celloracle.tfinfo'

In [43]:
reload(grnpruner)

<module 'src.grn_pruner' from '/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/grn_pruner.py'>

## 3.1. Base GRN based on cell oracle motif based prediction 

In [ ]:

# Step 6: Save outputs
print("Saving CellOracle outputs...")
co.save_tfinfo(paths['celloracle_h5'])
base_grn_df = co.save_dataframe(paths['celloracle_parquet'])

Saving CellOracle outputs...


In [25]:
# reload(grnpruner)
# # Step 6: Save outputs
# print("Saving CellOracle outputs...")
# co.save_tfinfo(paths['celloracle_h5'])
# base_grn_df = co.save_dataframe(paths['celloracle_parquet'])

print(f"\nBase GRN saved to: {paths['celloracle_parquet']}")
print(f"Base GRN shape: {base_grn_df.shape}")
print(f"TFs in base GRN: {base_grn_df.shape[1] - 2}")  # -2 for peak_id and gene columns

# ==============================================================================
# STEP 5: GRN PRUNING (CORRELATION + DE FILTERING)
# ==============================================================================

print("\n" + "="*80)
print("STEP 5: GRN PRUNING")
print("="*80)

# Get all TFs (E14 + E18 DE TFs)
all_tf_list = tf_df['Symbol'].tolist()
print(f"\nTotal DE TFs for pruning: {len(all_tf_list)}")

# Initialize pruner
pruner = grnpruner.GRNPruner(
    pkn_file=paths['celloracle_parquet'],
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    tf_list=all_tf_list,
    lfc_threshold=1.0,
    corr_threshold=0.7,
    qval_threshold=0.05
)

# Build stage-specific networks
e14_grn, e18_grn = pruner.build_networks()

# Save networks
e14_grn.to_csv(paths['e14_grn'], index=False)
e18_grn.to_csv(paths['e18_grn'], index=False)

print(f"\nE14 network saved to: {paths['e14_grn']}")
print(f"E18 network saved to: {paths['e18_grn']}")

# ==============================================================================
# STEP 6: SUMMARY
# ==============================================================================

print("\n" + "="*80)
print("PIPELINE COMPLETE - SUMMARY")
print("="*80)

summary = pd.DataFrame({
    'Step': [
        '1. DESeq2',
        '2. Consensus Peaks',
        '3. Enhancer Integration',
        '4. CellOracle Base GRN',
        '5. E14 TF Network',
        '6. E18 TF Network'
    ],
    'Count': [
        f"{len(de_tfs)} DE TFs",
        f"{len(consensus)} peaks",
        f"{len(target_genes)} genes with accessible enhancers",
        f"{base_grn_df.shape[1] - 2} TFs in base GRN",
        f"{len(e14_grn)} edges, {e14_grn['TF'].nunique()} TFs",
        f"{len(e18_grn)} edges, {e18_grn['TF'].nunique()} TFs"
    ]
})

print(summary.to_markdown(index=False))

print("\n" + "="*80)
print("OUTPUT FILES:")
print("="*80)
for key, path in paths.items():
    if 'output' in key or any(x in key for x in ['consensus', 'deseq', 'celloracle', 'grn']):
        print(f"  {key}: {path}")

print("\n" + "="*80)
print("NETWORK COMPARISON:")
print("="*80)

e14_edges = set(zip(e14_grn['TF'], e14_grn['target']))
e18_edges = set(zip(e18_grn['TF'], e18_grn['target']))

shared_edges = e14_edges & e18_edges
e14_specific = e14_edges - e18_edges
e18_specific = e18_edges - e14_edges

print(f"\nShared edges: {len(shared_edges)}")
print(f"E14-specific edges: {len(e14_specific)}")
print(f"E18-specific edges: {len(e18_specific)}")

print("\nTop 10 E14 hub TFs:")
print(e14_grn['TF'].value_counts().head(10))

print("\nTop 10 E18 hub TFs:")
print(e18_grn['TF'].value_counts().head(10))


Base GRN saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/celloracle_base_grn.parquet
Base GRN shape: (20783, 1141)
TFs in base GRN: 1139

STEP 5: GRN PRUNING

Total DE TFs for pruning: 1482
Loading data...
PKN: 20783 peaks, TF list: 1473 TFs

=== Converting PKN ===
Total edges: 3629655

=== Filtering to TF-TF ===
TF-TF edges: 191827

=== E14 Network ===
After DE filter: 172
Correlations computed: 168
Final E14: 59 edges, 14 TFs

=== E18 Network ===
After DE filter: 1351
Correlations computed: 1336
Final E18: 91 edges, 24 TFs

E14 network saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/E14_TF_network.csv
E18 network saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/E18_TF_network.csv

PIPELINE COMPLETE - SUMMARY
| Step                    | Count                                 |
|:------------------------|:-------------------------------

## 3.2. GRN based on chipseq data from SincMAT

In [ ]:


# # Option 1: CellOracle
# pruner = GRNPruner(
#     base_grn='celloracle_base_grn.parquet',
#     counts_file='counts.csv',
#     metadata_file='metadata.csv',
#     deseq_file='deseq.csv',
#     tf_list=['Sox2', 'Pax6', ...],
#     grn_format='auto'  # or 'celloracle'
# )

# Option 2: ChIP-seq edge list
chipseq_grn = pd.read_csv('/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/chip_seq/mouse_all_chip.csv')
prunerV2 = grnpruner.GRNPruner(
    base_grn=chipseq_grn,
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    tf_list=all_tf_list,
    grn_format='chipseq',
    lfc_threshold=1.0,
    corr_threshold=0.7,
    qval_threshold=0.05
)

# # Option 3: From CSV file
# pruner = GRNPruner(
#     base_grn='chipseq_edges.csv',
#     counts_file='counts.csv',
#     metadata_file='metadata.csv',
#     deseq_file='deseq.csv',
#     tf_list=['Sox2', 'Pax6', ...],
#     grn_format='auto'
# )
#
e14_grn_v2, e18_grn_v2 = prunerV2.build_networks()

Loading base GRN...
GRN format: chipseq
Loading RNA-seq data...
Base GRN loaded, TF list: 1473 TFs

=== Converting to edge list ===
ChIP-seq edges: 6946884
Total edges: 6946884

=== Filtering to TF-TF ===
TF-TF edges: 288546

=== E14 Network ===
After DE filter: 330
Correlations computed: 317
Final E14: 136 edges, 20 TFs

=== E18 Network ===
After DE filter: 1932
Correlations computed: 1898
Final E18: 147 edges, 36 TFs


In [33]:
e14_grn_v2

,TF,target,correlation,pval,qval
0,Tcf7l1,Neurog1,0.721212,0.018573,0.035256
1,Lef1,Neurog1,0.745455,0.013330,0.027619
2,Lef1,Tcf7l1,0.975758,0.000001,0.000093
3,Tcf7l1,Lef1,0.975758,0.000001,0.000093
4,Ebf2,Lef1,0.721212,0.018573,0.035256
...,...,...,...,...,...
290,Fezf2,Dmrt3,0.745455,0.013330,0.027619
291,Fezf2,Nr0b1,0.890909,0.000542,0.005055
299,Lhx9,Sox3,0.757576,0.011143,0.026761
305,Six4,Sall4,0.709091,0.021666,0.038803


In [34]:
e14_grn

,TF,target,correlation,pval,qval
0,E2f2,Hmga2,0.842424,0.002220,0.010656
4,Nhlh1,Otx1,0.878788,0.000814,0.004715
5,Nhlh2,Otx1,0.721212,0.018573,0.030591
12,Nhlh1,Six4,0.781818,0.007547,0.018375
13,Nhlh2,Six4,0.830303,0.002940,0.012349
15,Rorc,Six4,0.709091,0.021666,0.034338
16,E2f2,E2f3,0.903030,0.000344,0.002749
28,Lhx9,Neurog1,0.915152,0.000204,0.002454
30,Nhlh1,Neurog1,0.963636,0.000007,0.000410
31,Nhlh2,Neurog1,0.890909,0.000542,0.003795


# ==============================================================================
# 4 Establish GRN Hierarchies based on TF-Gene Networks
# ==============================================================================

In [41]:
def assign_hierarchical_layers_genomic_hybrid(grn_df, base_grn_motif, peak_annot, enhancer_overlap_df):
    """
    Hybrid approach with FIXED case handling.
    """
    import networkx as nx
    import numpy as np
    import pandas as pd
    
    # Extract enhancer peaks
    enhancer_overlap_df['peak_id'] = (
        enhancer_overlap_df['Chromosome'].astype(str) + '_' + 
        enhancer_overlap_df['Start_b'].astype(str) + '_' + 
        enhancer_overlap_df['End_b'].astype(str)
    )
    enhancer_peak_ids = set(enhancer_overlap_df['peak_id'].unique())
    
    def classify_peak(row):
        in_enhancer_db = row['peak_id'] in enhancer_peak_ids
        distance = row['distance_to_tss']
        
        # Enhancer must be EITHER:
        # 1. In EnhancerDB AND >1kb from TSS, OR
        # 2. Very distal (>50kb) regardless of EnhancerDB
        
        if in_enhancer_db and distance > 1000:
            return 'enhancer_db'
        elif distance < 1000:
            return 'promoter'
        elif distance < 10000:
            return 'proximal'
        elif distance < 50000:
            return 'distal'
        else:  # >50kb
            return 'enhancer_distal'
    
    peak_annot['peak_class'] = peak_annot.apply(classify_peak, axis=1)
    peak_annot['is_enhancer_like'] = peak_annot['peak_class'].isin([
        'enhancer_db', 'distal', 'enhancer_distal'
    ])
    
    print(f"\n=== Peak Classification (Hybrid) ===")
    print(f"Total unique peaks: {len(peak_annot)}")
    print(f"EnhancerDB: {len(peak_annot[peak_annot['peak_class'] == 'enhancer_db'])} ({len(peak_annot[peak_annot['peak_class'] == 'enhancer_db'])/len(peak_annot)*100:.1f}%)")
    print(f"Promoter: {len(peak_annot[peak_annot['peak_class'] == 'promoter'])} ({len(peak_annot[peak_annot['peak_class'] == 'promoter'])/len(peak_annot)*100:.1f}%)")
    print(f"Combined enhancer-like: {peak_annot['is_enhancer_like'].sum()} ({peak_annot['is_enhancer_like'].sum()/len(peak_annot)*100:.1f}%)")
    
    # Standardize TF names to match motif matrix
    grn_df = grn_df.copy()
    grn_df['TF'] = grn_df['TF'].str.capitalize()
    grn_df['target'] = grn_df['target'].str.capitalize()
    
    # FIXED: Case-insensitive mapping that returns original column name
    motif_col_map = {col.lower(): col for col in base_grn_motif.columns}
    
    # Calculate TF binding profiles
    tf_profiles = {}
    
    print(f"\n=== TF Binding Profiles ===")
    
    for tf in sorted(set(grn_df['TF'].unique()) | set(grn_df['target'].unique())):
        tf_lower = tf.lower()
        
        # FIXED: Use case-insensitive lookup
        if tf_lower not in motif_col_map:
            print(f"{tf}: NOT FOUND in motif matrix")
            tf_profiles[tf] = {
                'n_peaks': 0, 'n_enhancer_db': 0, 'n_distal': 0,
                'n_enhancer_like': 0, 'n_promoter': 0,
                'enhancer_like_ratio': 0, 'median_distance': 0,
                'upstream_score': 0
            }
            continue
        
        # Get actual column name (with correct case)
        motif_col = motif_col_map[tf_lower]
        
        # Get peaks where this TF binds
        tf_peaks = base_grn_motif[base_grn_motif[motif_col] == 1].index.tolist()
        
        if len(tf_peaks) == 0:
            print(f"{tf}: No binding peaks (motif found but no sites)")
            tf_profiles[tf] = {
                'n_peaks': 0, 'n_enhancer_db': 0, 'n_distal': 0,
                'n_enhancer_like': 0, 'n_promoter': 0,
                'enhancer_like_ratio': 0, 'median_distance': 0,
                'upstream_score': 0
            }
            continue
        
        tf_peak_info = peak_annot[peak_annot['peak_id'].isin(tf_peaks)]
        
        if len(tf_peak_info) == 0:
            print(f"{tf}: Peaks not in annotation")
            continue
        
        # Count by classification
        n_total = len(tf_peak_info)
        n_enhancer_db = len(tf_peak_info[tf_peak_info['peak_class'] == 'enhancer_db'])
        n_promoter = len(tf_peak_info[tf_peak_info['peak_class'] == 'promoter'])
        n_proximal = len(tf_peak_info[tf_peak_info['peak_class'] == 'proximal'])
        n_distal = len(tf_peak_info[tf_peak_info['peak_class'] == 'distal'])
        n_enhancer_distal = len(tf_peak_info[tf_peak_info['peak_class'] == 'enhancer_distal'])
        
        n_enhancer_like = tf_peak_info['is_enhancer_like'].sum()
        enhancer_like_ratio = n_enhancer_like / n_total
        
        median_dist = tf_peak_info['distance_to_tss'].median()
        
        upstream_score = (enhancer_like_ratio * 1000) + (median_dist / 100)
        
        tf_profiles[tf] = {
            'n_peaks': n_total,
            'n_enhancer_db': n_enhancer_db,
            'n_promoter': n_promoter,
            'n_proximal': n_proximal,
            'n_distal': n_distal,
            'n_enhancer_distal': n_enhancer_distal,
            'n_enhancer_like': n_enhancer_like,
            'enhancer_like_ratio': enhancer_like_ratio,
            'median_distance': median_dist,
            'upstream_score': upstream_score
        }
        
        print(f"{tf}: {n_total} peaks → {n_enhancer_like} enh-like ({enhancer_like_ratio:.1%}), median_dist={median_dist:.0f}bp")
    
    # Build graph
    G = nx.DiGraph()
    for _, row in grn_df.iterrows():
        source_score = tf_profiles.get(row['TF'], {}).get('upstream_score', 0)
        target_score = tf_profiles.get(row['target'], {}).get('upstream_score', 0)
        edge_weight = source_score - target_score
        G.add_edge(row['TF'], row['target'], weight=edge_weight, correlation=row['correlation'])
    
    # Break cycles
    G_dag = G.copy()
    removed_edges = []
    
    while not nx.is_directed_acyclic_graph(G_dag):
        try:
            cycle = nx.find_cycle(G_dag)
            min_weight = float('inf')
            edge_to_remove = None
            
            for u, v in cycle:
                weight = G_dag[u][v]['weight']
                if weight < min_weight:
                    min_weight = weight
                    edge_to_remove = (u, v)
            
            if edge_to_remove:
                removed_edges.append(edge_to_remove)
                G_dag.remove_edge(*edge_to_remove)
        except nx.NetworkXNoCycle:
            break
    
    print(f"\n=== Cycle Breaking ===")
    print(f"Removed {len(removed_edges)} edges:")
    for u, v in removed_edges:
        u_prof = tf_profiles.get(u, {})
        v_prof = tf_profiles.get(v, {})
        print(f"  {u} ({u_prof.get('enhancer_like_ratio', 0):.1%} enh-like) → "
              f"{v} ({v_prof.get('enhancer_like_ratio', 0):.1%} enh-like)")
    
    # Assign layers
    layers = {}
    for i, nodes in enumerate(nx.topological_generations(G_dag)):
        for node in nodes:
            layers[node] = i
    
    print(f"\n=== Layer Summary ===")
    layer_nodes = {}
    for node, layer in layers.items():
        if layer not in layer_nodes:
            layer_nodes[layer] = []
        layer_nodes[layer].append(node)
    
    for layer in sorted(layer_nodes.keys()):
        nodes = layer_nodes[layer]
        nodes_with_data = [n for n in nodes if tf_profiles.get(n, {}).get('n_peaks', 0) > 0]
        
        if nodes_with_data:
            avg_enh = np.mean([tf_profiles.get(n, {}).get('enhancer_like_ratio', 0) for n in nodes_with_data])
        else:
            avg_enh = 0
        
        print(f"\nLayer {layer}: {len(nodes)} TFs, avg enhancer-like ratio = {avg_enh:.1%}")
        for tf in sorted(nodes, key=lambda x: tf_profiles.get(x, {}).get('enhancer_like_ratio', 0), reverse=True):
            prof = tf_profiles.get(tf, {})
            if prof.get('n_peaks', 0) > 0:
                print(f"  {tf}: {prof.get('n_enhancer_like', 0)}/{prof.get('n_peaks', 0)} enh-like "
                      f"({prof.get('enhancer_like_ratio', 0):.1%}) "
                      f"[DB:{prof.get('n_enhancer_db', 0)}, Distal:{prof.get('n_distal', 0)+prof.get('n_enhancer_distal', 0)}]")
            else:
                print(f"  {tf}: NO DATA")
    
    return layers, tf_profiles

def plot_hierarchical_network_genomic(grn_df, base_grn_motif, peak_annot, enhancer_overlap_df,
                                      stage_name, output_file, figsize=(30, 20)):
    """
    Plot hierarchical network with genomic coordinate-based layers and enhancer ratio coloring.
    """
    import matplotlib.pyplot as plt
    import networkx as nx
    import numpy as np
    import pandas as pd
    
    # Get layers and profiles
    layers, tf_profiles = assign_hierarchical_layers_genomic_hybrid(
        grn_df, base_grn_motif, peak_annot, enhancer_overlap_df
    )
    
    # Build graph for visualization (include all edges from original network)
    G = nx.from_pandas_edgelist(
        grn_df[grn_df['TF'] != grn_df['target']], 
        source='TF', 
        target='target', 
        create_using=nx.DiGraph()
    )
    
    print(f"\n=== Building {stage_name} Hierarchical Layout ===")
    print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    
    # Group nodes by layer
    layer_nodes = {}
    for node, layer in layers.items():
        if layer not in layer_nodes:
            layer_nodes[layer] = []
        layer_nodes[layer].append(node)
    
    # Calculate positions
    pos = {}
    for layer, nodes in layer_nodes.items():
        # Sort within layer by enhancer ratio (left to right: high to low)
        nodes_sorted = sorted(
            nodes, 
            key=lambda x: tf_profiles.get(x, {}).get('enhancer_like_ratio', 0), 
            reverse=True
        )
        
        n = len(nodes_sorted)
        for i, node in enumerate(nodes_sorted):
            x = (i - n/2) * 3  # Spacing between nodes
            y = -layer * 4     # Spacing between layers
            pos[node] = (x, y)
    
    # Prepare node colors based on enhancer ratio
    node_colors = []
    node_sizes = []
    
    for node in G.nodes():
        prof = tf_profiles.get(node, {})
        enh_ratio = prof.get('enhancer_like_ratio', 0)
        n_peaks = prof.get('n_peaks', 0)
        
        node_colors.append(enh_ratio)
        
        # Size based on number of peaks (log scale)
        if n_peaks > 0:
            size = 300 + np.log10(n_peaks + 1) * 300
        else:
            size = 300  # Default size for TFs without data
        node_sizes.append(size)
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Draw nodes
    nodes_collection = nx.draw_networkx_nodes(
        G, pos, 
        node_size=node_sizes,
        node_color=node_colors, 
        cmap='RdYlBu_r',  # Red (high enhancer) to Blue (low enhancer)
        vmin=0, 
        vmax=0.6,  # Scale to show differences
        alpha=0.9,
        edgecolors='black',
        linewidths=1.5
    )
    
    # Draw edges with varying transparency based on correlation
    edge_colors = []
    edge_widths = []
    
    for u, v in G.edges():
        # Get correlation from original grn_df
        edge_data = grn_df[(grn_df['TF'] == u) & (grn_df['target'] == v)]
        if len(edge_data) > 0:
            corr = edge_data['correlation'].values[0]
            # Map correlation to alpha (0.8-1.0 -> 0.2-0.8)
            alpha_val = 0.2 + (corr - 0.8) * 3  # Scale 0.8-1.0 to 0.2-0.8
            edge_colors.append((0.3, 0.3, 0.3, max(0.2, min(0.8, alpha_val))))
            edge_widths.append(1 + corr * 2)  # Thicker for higher correlation
        else:
            edge_colors.append((0.3, 0.3, 0.3, 0.3))
            edge_widths.append(1)
    
    nx.draw_networkx_edges(
        G, pos, 
        edge_color=edge_colors,
        width=edge_widths,
        arrows=True, 
        arrowsize=15,
        arrowstyle='-|>',
        connectionstyle='arc3,rad=0.1',
        node_size=node_sizes
    )
    
    # Draw labels
    # Adjust font size based on number of nodes
    if G.number_of_nodes() > 40:
        font_size = 8
    elif G.number_of_nodes() > 25:
        font_size = 10
    else:
        font_size = 12
    
    # Add labels with background for readability
    for node, (x, y) in pos.items():
        prof = tf_profiles.get(node, {})
        enh_ratio = prof.get('enhancer_like_ratio', 0)
        n_peaks = prof.get('n_peaks', 0)
        
        # Label with enhancer percentage
        if n_peaks > 0:
            label = f"{node}\n{enh_ratio:.0%}"
        else:
            label = f"{node}\n(no data)"
        
        ax.text(
            x, y, label,
            fontsize=font_size,
            fontweight='bold',
            ha='center',
            va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                     edgecolor='none', alpha=0.8)
        )
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(
        cmap='RdYlBu_r', 
        norm=plt.Normalize(vmin=0, vmax=0.6)
    )
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('Enhancer-like Binding Ratio', rotation=270, labelpad=25, fontsize=14)
    cbar.set_ticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
    cbar.set_ticklabels(['0%', '10%', '20%', '30%', '40%', '50%', '60%'])
    
    # Add layer labels on the left
    for layer in sorted(layer_nodes.keys()):
        nodes = layer_nodes[layer]
        if nodes:
            y_pos = -layer * 4
            ax.text(
                min([pos[n][0] for n in nodes]) - 5, y_pos,
                f'Layer {layer}',
                fontsize=14,
                fontweight='bold',
                va='center',
                ha='right',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.7)
            )
    
    # Title and formatting
    title = f'{stage_name} TF Regulatory Network - Genomic Hierarchy\n'
    title += f'({G.number_of_nodes()} TFs, {G.number_of_edges()} regulatory links)\n'
    title += f'Color: Enhancer binding ratio | Size: Number of binding peaks | Edge: Correlation strength'
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')
    plt.tight_layout()
    
    # Save
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved to: {output_file}")
    plt.show()
    
    # Print summary statistics
    print(f"\n=== {stage_name} Network Statistics ===")
    print(f"Total TFs: {G.number_of_nodes()}")
    print(f"Total edges: {G.number_of_edges()}")
    print(f"Layers: {len(layer_nodes)}")
    
    # Calculate average out-degree per layer
    print(f"\n=== Layer Statistics ===")
    for layer in sorted(layer_nodes.keys()):
        nodes = layer_nodes[layer]
        out_degrees = [G.out_degree(n) for n in nodes]
        in_degrees = [G.in_degree(n) for n in nodes]
        avg_enh = np.mean([tf_profiles.get(n, {}).get('enhancer_like_ratio', 0) 
                          for n in nodes if tf_profiles.get(n, {}).get('n_peaks', 0) > 0])
        
        print(f"\nLayer {layer}: {len(nodes)} TFs")
        print(f"  Avg out-degree: {np.mean(out_degrees):.1f}")
        print(f"  Avg in-degree: {np.mean(in_degrees):.1f}")
        print(f"  Avg enhancer ratio: {avg_enh:.1%}")
        print(f"  Hub TFs (out-degree > 5): {[n for n in nodes if G.out_degree(n) > 5]}")
    
    return G, layers, tf_profiles, pos



In [42]:
# Keep only unique peaks (first occurrence)
peak_annot_unique = peak_annot.drop_duplicates(subset='peak_id', keep='first')
print(f"Unique peaks in peak_annot: {len(peak_annot_unique)}")
# Set index properly
base_grn_motif_indexed = base_grn_df.set_index('peak_id')
base_grn_motif_indexed = base_grn_motif_indexed.drop(columns=['gene_short_name'])

# Run with fixed case handling
layers_e14, profiles_e14 = assign_hierarchical_layers_genomic_hybrid(
    grn_df=e14_grn_v2,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_overlap_df=overlaps_df
)

# Run with fixed case handling
layers_e18, profiles_e18 = assign_hierarchical_layers_genomic_hybrid(
    grn_df=e18_grn_v2,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_overlap_df=overlaps_df
)



# Plot E14 network
G_e14, layers_e14, profiles_e14, pos_e14 = plot_hierarchical_network_genomic(
    grn_df=e14_grn_v2,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_overlap_df=overlaps_df,
    stage_name='E14',
    output_file='e14_genomic_hierarchy_v2.png',
    figsize=(30, 20)
)

# Plot E18 network
G_e18, layers_e18, profiles_e18, pos_e18 = plot_hierarchical_network_genomic(
    grn_df=e18_grn_v2,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_overlap_df=overlaps_df,
    stage_name='E18',
    output_file='e18_genomic_hierarchy_v2.png',
    figsize=(30, 20)
)

Unique peaks in peak_annot: 17436

=== Peak Classification (Hybrid) ===
Total unique peaks: 17436
EnhancerDB: 1235 (7.1%)
Promoter: 13639 (78.2%)
Combined enhancer-like: 2743 (15.7%)

=== TF Binding Profiles ===
Bcl11b: NOT FOUND in motif matrix
Dmrt2: 109 peaks → 21 enh-like (19.3%), median_dist=275bp
Dmrt3: 69 peaks → 9 enh-like (13.0%), median_dist=246bp
E2f2: 13823 peaks → 1982 enh-like (14.3%), median_dist=200bp
E2f3: 13746 peaks → 1981 enh-like (14.4%), median_dist=200bp
Ebf2: NOT FOUND in motif matrix
Fezf2: 249 peaks → 46 enh-like (18.5%), median_dist=272bp
Hmga1: 59 peaks → 18 enh-like (30.5%), median_dist=290bp
Hmga2: 35 peaks → 13 enh-like (37.1%), median_dist=300bp
Isl2: 150 peaks → 30 enh-like (20.0%), median_dist=291bp
Lef1: 133 peaks → 28 enh-like (21.1%), median_dist=328bp
Lhx1: 935 peaks → 176 enh-like (18.8%), median_dist=288bp
Lhx9: 935 peaks → 176 enh-like (18.8%), median_dist=298bp
Lin28a: NOT FOUND in motif matrix
Lin28b: NOT FOUND in motif matrix
Neurog1: 1383 pe

/home/users/adhal/micromamba/envs/scrna_target_idf/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Peak Classification (Hybrid) ===
Total unique peaks: 17436
EnhancerDB: 1235 (7.1%)
Promoter: 13639 (78.2%)
Combined enhancer-like: 2743 (15.7%)

=== TF Binding Profiles ===
Ahr: 3862 peaks → 591 enh-like (15.3%), median_dist=225bp
Csdc2: NOT FOUND in motif matrix
Elk3: 2191 peaks → 292 enh-like (13.3%), median_dist=175bp
Erg: 3599 peaks → 497 enh-like (13.8%), median_dist=188bp
Ets1: 10653 peaks → 1577 enh-like (14.8%), median_dist=209bp
Etv1: 2960 peaks → 375 enh-like (12.7%), median_dist=175bp
Etv4: 2625 peaks → 365 enh-like (13.9%), median_dist=184bp
Fli1: 6617 peaks → 1005 enh-like (15.2%), median_dist=209bp
Fos: 3299 peaks → 504 enh-like (15.3%), median_dist=195bp
Foxc1: 15160 peaks → 2356 enh-like (15.5%), median_dist=214bp
Foxf2: 15285 peaks → 2375 enh-like (15.5%), median_dist=214bp
Foxj1: 230 peaks → 41 enh-like (17.8%), median_dist=226bp
Foxl2: 15285 peaks → 2375 enh-like (15.5%), median_dist=214bp
Foxn4: NOT FOUND in motif matrix
Foxo1: 857 peaks → 139 enh-like (16.2%),

In [108]:
def expand_grn_with_de_genes(base_grn_motif, peak_annot, de_genes, de_tfs, 
                              expression_e14, expression_e18, 
                              corr_threshold=0.8, stage='E14'):
    """
    Expand TF-TF network to include TF → DE gene links.
    """
    import pandas as pd
    import numpy as np
    from scipy.stats import pearsonr
    
    # Select expression for this stage
    expression = expression_e14 if stage == 'E14' else expression_e18
    
    # Convert de_genes to set for faster lookup
    de_genes_set = set(de_genes)
    
    # Get genes present in BOTH expression matrix AND de_genes list
    available_genes = set(expression.index) & de_genes_set
    available_tfs = set(expression.index) & set(de_tfs)
    
    print(f"Expanding GRN for {stage}...")
    print(f"DE genes: {len(de_genes_set)}")
    print(f"Genes in expression: {len(expression.index)}")
    print(f"Overlapping genes: {len(available_genes)}")
    print(f"TFs in expression: {len(available_tfs)}")
    print(f"Scanning {len(available_tfs)} TFs × {len(available_genes)} DE genes")
    
    edges = []
    
    for tf in available_tfs:
        # Check if TF in motif matrix (case-insensitive)
        tf_lower = tf.lower()
        motif_col_map = {col.lower(): col for col in base_grn_motif.columns}
        
        if tf_lower not in motif_col_map:
            continue
        
        motif_col = motif_col_map[tf_lower]
        
        # Get peaks where TF binds
        tf_peaks = base_grn_motif[base_grn_motif[motif_col] == 1].index.tolist()
        
        # Get genes linked to these peaks
        tf_peak_genes = peak_annot[peak_annot['peak_id'].isin(tf_peaks)]['gene_short_name'].unique()
        
        # Keep only DE genes that are in expression matrix
        tf_de_targets = [g for g in tf_peak_genes if g in available_genes]
        
        for target in tf_de_targets:
            # Calculate correlation
            try:
                corr, pval = pearsonr(
                    expression.loc[tf].values,
                    expression.loc[target].values
                )
            except Exception as e:
                continue
            
            if abs(corr) >= corr_threshold:
                # Count peaks regulating this target
                n_peaks = len(peak_annot[
                    (peak_annot['peak_id'].isin(tf_peaks)) & 
                    (peak_annot['gene_short_name'] == target)
                ])
                
                edges.append({
                    'TF': tf,
                    'target': target,
                    'correlation': corr,
                    'target_is_tf': target in available_tfs,
                    'n_peaks': n_peaks
                })
        
        if len(edges) > 0 and len(edges) % 100 == 0:
            print(f"  {tf}: {len(edges)} total edges so far...")
    
    expanded_grn = pd.DataFrame(edges)
    
    print(f"\nExpanded GRN: {len(expanded_grn)} edges")
    print(f"  TF → TF: {expanded_grn['target_is_tf'].sum()}")
    print(f"  TF → Gene: {(~expanded_grn['target_is_tf']).sum()}")
    
    return expanded_grn



def calculate_tf_regulatory_enhancer_ratio(tf, expanded_grn, base_grn_motif, 
                                           peak_annot, enhancer_peak_ids):
    """
    Calculate enhancer ratio based on ACTUAL regulatory targets.
    
    Args:
        tf: TF name
        expanded_grn: Expanded network from expand_grn_with_de_genes()
        base_grn_motif: Motif matrix
        peak_annot: Peak annotations with peak_class column
        enhancer_peak_ids: Set of enhancer peak IDs
    
    Returns:
        dict with regulatory statistics
    """
    import numpy as np
    
    # Get all targets of this TF
    tf_targets = expanded_grn[expanded_grn['TF'] == tf]['target'].tolist()
    
    if len(tf_targets) == 0:
        return {
            'n_targets': 0,
            'n_regulatory_peaks': 0,
            'n_enhancer_peaks': 0,
            'n_promoter_peaks': 0,
            'regulatory_enh_ratio': 0,
            'median_distance': 0
        }
    
    # Get TF binding peaks
    if tf not in base_grn_motif.columns:
        return {'n_targets': 0, 'n_regulatory_peaks': 0, 'n_enhancer_peaks': 0,
                'n_promoter_peaks': 0, 'regulatory_enh_ratio': 0, 'median_distance': 0}
    
    tf_peaks = base_grn_motif[base_grn_motif[tf] == 1].index.tolist()
    
    # Get peaks that regulate these specific targets
    regulatory_peaks = peak_annot[
        (peak_annot['peak_id'].isin(tf_peaks)) &
        (peak_annot['gene_short_name'].isin(tf_targets))
    ]
    
    if len(regulatory_peaks) == 0:
        return {'n_targets': len(tf_targets), 'n_regulatory_peaks': 0, 
                'n_enhancer_peaks': 0, 'n_promoter_peaks': 0,
                'regulatory_enh_ratio': 0, 'median_distance': 0}
    
    # Count enhancer vs promoter peaks
    n_enhancer = regulatory_peaks['is_enhancer_like'].sum()
    n_promoter = (regulatory_peaks['peak_class'] == 'promoter').sum()
    n_total = len(regulatory_peaks)
    
    regulatory_enh_ratio = n_enhancer / n_total
    median_distance = regulatory_peaks['distance_to_tss'].median()
    
    return {
        'n_targets': len(tf_targets),
        'n_regulatory_peaks': n_total,
        'n_enhancer_peaks': n_enhancer,
        'n_promoter_peaks': n_promoter,
        'regulatory_enh_ratio': regulatory_enh_ratio,
        'median_distance': median_distance
    }


def compare_global_vs_regulatory_ratios(de_tfs, expanded_grn, base_grn_motif,
                                       peak_annot, enhancer_peak_ids, tf_profiles):
    """
    Compare global enhancer ratio vs regulatory enhancer ratio.
    
    Returns:
        DataFrame comparing both metrics
    """
    import pandas as pd
    
    comparison = []
    
    for tf in de_tfs:
        # Global ratio (from all peaks)
        global_prof = tf_profiles.get(tf, {})
        global_ratio = global_prof.get('enhancer_like_ratio', 0)
        global_peaks = global_prof.get('n_peaks', 0)
        
        # Regulatory ratio (from target-specific peaks)
        reg_stats = calculate_tf_regulatory_enhancer_ratio(
            tf, expanded_grn, base_grn_motif, peak_annot, enhancer_peak_ids
        )
        
        comparison.append({
            'TF': tf,
            'n_targets': reg_stats['n_targets'],
            'global_peaks': global_peaks,
            'regulatory_peaks': reg_stats['n_regulatory_peaks'],
            'global_enh_ratio': global_ratio,
            'regulatory_enh_ratio': reg_stats['regulatory_enh_ratio'],
            'difference': reg_stats['regulatory_enh_ratio'] - global_ratio
        })
    
    df = pd.DataFrame(comparison)
    df = df.sort_values('regulatory_enh_ratio', ascending=False)
    
    return df



In [112]:
pruner_counts = pd.read_csv(paths['counts'], sep=';', header=0).iloc[:, 1:]
pruner_counts = pruner_counts.set_index(pruner_counts.columns[0])

In [130]:
def prepare_expression_and_de_genes(pruner_metadata, pruner_deseq, pruner_counts, 
                                    stage='E14', padj_threshold=0.05, logfc_threshold=1.0):
    """
    Extract expression matrix and DE gene list for a specific stage.
    """
    import pandas as pd
    
    # 1. Counts already has genes as index - use directly
    counts = pruner_counts.dropna()
    
    print(f"Counts: {counts.shape[0]} genes × {counts.shape[1]} samples")
    
    # 2. Get samples for this stage (exclude outliers)
    outliers = ['MUC9939', 'MUC9940', 'MUC9914']
    stage_samples = pruner_metadata[
        (pruner_metadata['Stage'] == stage) & 
        (pruner_metadata['Marker'] == 'Progenitors') &
        (~pruner_metadata['SampleID'].isin(outliers))
    ]['SampleID'].tolist()
    
    print(f"Stage {stage}: {len(stage_samples)} samples")
    
    # 3. Extract expression for this stage
    expression = counts[stage_samples]
    
    print(f"Expression matrix: {expression.shape[0]} genes × {expression.shape[1]} samples")
    
    # 4. Get DE genes for this stage
    de_mask = (
        (pruner_deseq['padj'] < padj_threshold) &
        (abs(pruner_deseq['log2FoldChange']) > logfc_threshold) &
        (pruner_deseq['direction'] == stage)
    )
    
    de_results = pruner_deseq[de_mask].copy()
    
    de_genes = de_results['symbol'].dropna().tolist()
    de_tfs = de_results[de_results['is_TF'] == True]['symbol'].dropna().tolist()
    
    print(f"DE genes: {len(de_genes)} total, {len(de_tfs)} TFs")
    
    # 5. Verify overlap
    genes_in_expression = set(expression.index)
    de_in_expression = set(de_genes) & genes_in_expression
    tfs_in_expression = set(de_tfs) & genes_in_expression
    
    print(f"DE genes in expression: {len(de_in_expression)}/{len(de_genes)}")
    print(f"DE TFs in expression: {len(tfs_in_expression)}/{len(de_tfs)}")
    
    return expression, de_genes, de_tfs


# Usage:
expression_e14, de_genes_e14, de_tfs_e14 = prepare_expression_and_de_genes(
    pruner_metadata=pruner.metadata,
    pruner_deseq=pruner.deseq,
    pruner_counts=pruner.counts,
    stage='E14'
)

expression_e18, de_genes_e18, de_tfs_e18 = prepare_expression_and_de_genes(
    pruner_metadata=pruner.metadata,
    pruner_deseq=pruner.deseq,
    pruner_counts=pruner.counts,
    stage='E18'
)

print("\nReady for GRN expansion:")
print(f"E14: {expression_e14.shape}, {len(de_genes_e14)} DE genes")
print(f"E18: {expression_e18.shape}, {len(de_genes_e18)} DE genes")

# It's the set of peak IDs that overlap EnhancerDB
# You create it from your enhancer overlap dataframe:

enhancer_peak_ids = set(overlaps_df['peak_id'].unique())

print(f"Enhancer peaks: {len(enhancer_peak_ids)}")
# Should print: Enhancer peaks: 6309 (or similar)
# 1. Expand network
expanded_e14 = expand_grn_with_de_genes(
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    de_genes=de_genes_e14,  # Your DE gene list
    de_tfs=e14_grn['TF'].unique().tolist(),
    expression_e14=expression_e14,  # Your expression matrix
    expression_e18=expression_e18,
    corr_threshold=0.8,
    stage='E14'
)

# 2. Calculate regulatory ratios
print("\nExample: Nhlh1 regulatory profile")
nhlh1_reg = calculate_tf_regulatory_enhancer_ratio(
    'Nhlh1', 
    expanded_e14,
    base_grn_motif_indexed,
    peak_annot_unique,
    enhancer_peak_ids
)
print(nhlh1_reg)

# 3. Compare global vs regulatory
comparison_df = compare_global_vs_regulatory_ratios(
    de_tfs=e14_grn['TF'].unique().tolist(),
    expanded_grn=expanded_e14,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_peak_ids=enhancer_peak_ids,
    tf_profiles=profiles_e14
)

print("\nGlobal vs Regulatory Enhancer Ratios:")
print(comparison_df[['TF', 'n_targets', 'global_enh_ratio', 'regulatory_enh_ratio', 'difference']])

Counts: 28655 genes × 36 samples
Stage E14: 10 samples
Expression matrix: 28655 genes × 10 samples
DE genes: 2012 total, 83 TFs
DE genes in expression: 2011/2012
DE TFs in expression: 83/83
Counts: 28655 genes × 36 samples
Stage E18: 10 samples
Expression matrix: 28655 genes × 10 samples
DE genes: 255 total, 39 TFs
DE genes in expression: 255/255
DE TFs in expression: 39/39

Ready for GRN expansion:
E14: (28655, 10), 2012 DE genes
E18: (28655, 10), 255 DE genes
Enhancer peaks: 9106
Expanding GRN for E14...
DE genes: 2011
Genes in expression: 28655
Overlapping genes: 2011
TFs in expression: 11
Scanning 11 TFs × 2011 DE genes

Expanded GRN: 804 edges
  TF → TF: 0
  TF → Gene: 804

Example: Nhlh1 regulatory profile
{'n_targets': 126, 'n_regulatory_peaks': 131, 'n_enhancer_peaks': 22, 'n_promoter_peaks': 105, 'regulatory_enh_ratio': 0.16793893129770993, 'median_distance': 265.0}

Global vs Regulatory Enhancer Ratios:
         TF  n_targets  global_enh_ratio  regulatory_enh_ratio  differenc

In [131]:
# Check what you actually have
print(f"de_tfs_e14 length: {len(de_tfs_e14)}")
print(f"First 10: {de_tfs_e14[:10]}")

de_tfs_e14 length: 83
First 10: ['Csdc2', 'Creb3l2', 'Egr1', 'Etv4', 'Dbx2', 'Klf9', 'Nfatc1', 'Sox10', 'Foxo1', 'Zbtb7c']


In [119]:
# Debug: Check why so many have 0% regulatory enhancer ratio
def debug_regulatory_ratios(tf, expanded_grn, base_grn_motif, peak_annot):
    """Debug why a TF has 0% regulatory enhancer ratio."""
    
    # Get targets
    targets = expanded_grn[expanded_grn['TF'] == tf]['target'].tolist()
    print(f"\n{tf}: {len(targets)} targets")
    print(f"Targets: {targets[:5]}...")  # First 5
    
    # Get TF peaks
    tf_col = [c for c in base_grn_motif.columns if c.lower() == tf.lower()]
    if not tf_col:
        print(f"  → NOT in motif matrix")
        return
    
    tf_peaks = base_grn_motif[base_grn_motif[tf_col[0]] == 1].index.tolist()
    print(f"  Total TF peaks: {len(tf_peaks)}")
    
    # Get regulatory peaks
    reg_peaks = peak_annot[
        (peak_annot['peak_id'].isin(tf_peaks)) &
        (peak_annot['gene_short_name'].isin(targets))
    ]
    
    print(f"  Regulatory peaks: {len(reg_peaks)}")
    
    if len(reg_peaks) > 0:
        print(f"  Peak classes:")
        print(reg_peaks['peak_class'].value_counts())
        print(f"  Enhancer-like: {reg_peaks['is_enhancer_like'].sum()}")
    else:
        print(f"  → NO regulatory peaks found!")
        # Check if targets are in peak_annot at all
        targets_in_annot = peak_annot[peak_annot['gene_short_name'].isin(targets)]['gene_short_name'].unique()
        print(f"  Targets in peak_annot: {len(targets_in_annot)}/{len(targets)}")

# Test on a few TFs
for tf in ['Bcl11b', 'Sall4', 'Nr0b1', 'Rorc', 'Nr4a2']:
    debug_regulatory_ratios(tf, expanded_e14, base_grn_motif_indexed, peak_annot_unique)


Bcl11b: 3 targets
Targets: ['Dio2', 'Nmi', 'Csf1']...
  Total TF peaks: 649
  Regulatory peaks: 3
  Peak classes:
peak_class
promoter    3
Name: count, dtype: int64
  Enhancer-like: 0

Sall4: 14 targets
Targets: ['Plekhg1', 'Hey2', 'Stox1', 'Ppp1r3g', 'Tst']...
  Total TF peaks: 5225
  Regulatory peaks: 14
  Peak classes:
peak_class
promoter    12
proximal     2
Name: count, dtype: int64
  Enhancer-like: 0

Nr0b1: 0 targets
Targets: []...
  Total TF peaks: 1723
  Regulatory peaks: 0
  → NO regulatory peaks found!
  Targets in peak_annot: 0/0

Rorc: 2 targets
Targets: ['Cbs', 'Kif6']...
  Total TF peaks: 782
  Regulatory peaks: 2
  Peak classes:
peak_class
promoter    2
Name: count, dtype: int64
  Enhancer-like: 0

Nr4a2: 4 targets
Targets: ['Tmcc3', 'Spata6l', 'Frem1', 'Cav2']...
  Total TF peaks: 1887
  Regulatory peaks: 4
  Peak classes:
peak_class
enhancer_distal    1
proximal           1
distal             1
promoter           1
Name: count, dtype: int64
  Enhancer-like: 2


In [136]:
def get_de_genes_with_enhancers(de_genes, enhancer_overlap_df):
    """
    Get DE genes that have enhancers in EnhancerDB.
    Returns enhanced peak→gene links.
    """
    
    # EnhancerDB has true enhancer→gene links
    enhancer_genes = enhancer_overlap_df[
        enhancer_overlap_df['symbol'].isin(de_genes)
    ]['symbol'].unique()
    
    print(f"DE genes: {len(de_genes)}")
    print(f"DE genes with EnhancerDB enhancers: {len(enhancer_genes)} ({len(enhancer_genes)/len(de_genes)*100:.1f}%)")
    
    # Create enhancer→gene mapping
    enhancer_gene_links = enhancer_overlap_df[
        enhancer_overlap_df['symbol'].isin(de_genes)
    ][['peak_id', 'symbol']].copy()
    
    enhancer_gene_links.columns = ['peak_id', 'gene_short_name']
    
    return enhancer_gene_links, enhancer_genes


def calculate_tf_regulatory_enhancer_ratio_v2(tf, expanded_grn, base_grn_motif, 
                                              peak_annot, enhancer_gene_links):
    """
    Calculate regulatory enhancer ratio using BOTH:
    1. CellOracle peak→gene links (for all peaks)
    2. EnhancerDB enhancer→gene links (for enhancer peaks)
    """
    import numpy as np
    
    # Get targets
    targets = expanded_grn[expanded_grn['TF'] == tf]['target'].tolist()
    
    if len(targets) == 0:
        return {'n_targets': 0, 'n_regulatory_peaks': 0, 'n_enhancer_peaks': 0,
                'n_promoter_peaks': 0, 'regulatory_enh_ratio': 0}
    
    # Get TF peaks
    tf_col = [c for c in base_grn_motif.columns if c.lower() == tf.lower()]
    if not tf_col:
        return {'n_targets': 0}
    
    tf_peaks = base_grn_motif[base_grn_motif[tf_col[0]] == 1].index.tolist()
    
    # Method 1: CellOracle links (all peaks)
    celloracle_peaks = peak_annot[
        (peak_annot['peak_id'].isin(tf_peaks)) &
        (peak_annot['gene_short_name'].isin(targets))
    ]
    
    # Method 2: EnhancerDB links (enhancer peaks only)
    enhancerdb_peaks = enhancer_gene_links[
        (enhancer_gene_links['peak_id'].isin(tf_peaks)) &
        (enhancer_gene_links['gene_short_name'].isin(targets))
    ]
    
    # Combine: Use EnhancerDB where available, CellOracle otherwise
    all_peak_ids = set(celloracle_peaks['peak_id']) | set(enhancerdb_peaks['peak_id'])
    
    # Classify peaks
    n_enhancer = len(enhancerdb_peaks)  # EnhancerDB = true enhancers
    n_promoter = len(celloracle_peaks[
        (celloracle_peaks['peak_class'] == 'promoter') &
        (~celloracle_peaks['peak_id'].isin(enhancerdb_peaks['peak_id']))
    ])
    n_total = len(all_peak_ids)
    
    if n_total == 0:
        return {'n_targets': len(targets), 'n_regulatory_peaks': 0,
                'n_enhancer_peaks': 0, 'n_promoter_peaks': 0,
                'regulatory_enh_ratio': 0}
    
    regulatory_enh_ratio = n_enhancer / n_total
    
    return {
        'n_targets': len(targets),
        'n_regulatory_peaks': n_total,
        'n_enhancer_peaks': n_enhancer,
        'n_promoter_peaks': n_promoter,
        'regulatory_enh_ratio': regulatory_enh_ratio,
        'source_celloracle': len(celloracle_peaks),
        'source_enhancerdb': len(enhancerdb_peaks)
    }


# Usage:
enhancer_gene_links, de_genes_with_enh = get_de_genes_with_enhancers(
    de_genes_e14, 
    overlaps_df
)

# Recalculate with enhanced links
for tf in e14_grn['TF'].unique().tolist():
    stats_v2 = calculate_tf_regulatory_enhancer_ratio_v2(
        tf, expanded_e14, base_grn_motif_indexed,
        peak_annot_unique, enhancer_gene_links
    )
    print(f"{tf}: {stats_v2}")

DE genes: 2012
DE genes with EnhancerDB enhancers: 1163 (57.8%)
E2f2: {'n_targets': 306, 'n_regulatory_peaks': 789, 'n_enhancer_peaks': 786, 'n_promoter_peaks': 188, 'regulatory_enh_ratio': 0.9961977186311787, 'source_celloracle': 329, 'source_enhancerdb': 786}
Nhlh1: {'n_targets': 126, 'n_regulatory_peaks': 209, 'n_enhancer_peaks': 171, 'n_promoter_peaks': 76, 'regulatory_enh_ratio': 0.8181818181818182, 'source_celloracle': 131, 'source_enhancerdb': 171}
Nhlh2: {'n_targets': 154, 'n_regulatory_peaks': 261, 'n_enhancer_peaks': 220, 'n_promoter_peaks': 95, 'regulatory_enh_ratio': 0.842911877394636, 'source_celloracle': 159, 'source_enhancerdb': 220}
Bcl11b: {'n_targets': 12, 'n_regulatory_peaks': 13, 'n_enhancer_peaks': 3, 'n_promoter_peaks': 9, 'regulatory_enh_ratio': 0.23076923076923078, 'source_celloracle': 12, 'source_enhancerdb': 3}
E2f3: {'n_targets': 97, 'n_regulatory_peaks': 264, 'n_enhancer_peaks': 256, 'n_promoter_peaks': 65, 'regulatory_enh_ratio': 0.9696969696969697, 'source

In [135]:
expanded_e14

,TF,target,correlation,target_is_tf,n_peaks
0,Nr0b1,Hivep2,0.953159,False,2
1,Nr0b1,Mcidas,0.811614,False,1
2,Nr0b1,Cacna1d,0.975089,False,1
3,Nr0b1,Ncald,0.904902,False,1
4,Nr0b1,Tst,0.874074,False,1
...,...,...,...,...,...
799,E2f3,Igsf1,0.822382,False,1
800,E2f3,Ccdc160,0.896156,False,1
801,E2f3,Tmem47,0.865108,False,1
802,E2f3,Eda,0.894360,False,1


# ==============================================================================
# 5. TRIPLET SYNERGY USING O Information
# ==============================================================================

In [37]:
"""
Greedy O-information maximization starting with triplets.
"""

import numpy as np
import pandas as pd
from scipy.stats import entropy
from itertools import combinations
from tqdm import tqdm
from sklearn.metrics import mutual_info_score


class GreedyTFCombinationFinder:
    """Find optimal TF combinations starting with triplets."""
    
    def __init__(self, grn_df, counts_file, metadata_file, deseq_file, stage, n_bins=3):
        """
        Args:
            grn_df: GRN DataFrame with [TF, target, ...]
            counts_file, metadata_file, deseq_file: Data files
            stage: 'E14' or 'E18'
            n_bins: Number of bins for discretization
        """
        self.grn = grn_df
        self.stage = stage
        self.n_bins = n_bins
        
        # Load expression data
        counts = pd.read_csv(counts_file, sep=';', header=0).iloc[:, 1:]
        counts = counts.set_index(counts.columns[0])
        
        # Map to symbols
        deseq = pd.read_csv(deseq_file, index_col=0)
        ensembl_to_symbol = dict(zip(deseq['gene_id'], deseq['symbol']))
        counts.index = counts.index.map(lambda x: ensembl_to_symbol.get(x, x))
        counts = counts[~counts.index.duplicated(keep='first')]
        
        metadata = pd.read_csv(metadata_file, sep=';')
        
        # Get stage samples
        mask = (metadata['Stage'] == stage) & (metadata['Marker'] == 'Progenitors')
        samples = metadata[mask]['SampleID'].tolist()
        
        # Normalize
        expr = counts[samples]
        cpm = expr.div(expr.sum(axis=0), axis=1) * 1e6
        self.expr = np.log2(cpm + 1)
        
        print(f"Loaded expression for {stage}: {self.expr.shape[0]} genes, {self.expr.shape[1]} samples")
    
    def _discretize(self, x):
        """Discretize continuous values into bins."""
        return pd.qcut(x, q=self.n_bins, labels=False, duplicates='drop')
    
    def _multi_information(self, *variables):
        """
        Calculate multi-information (total correlation) for n variables.
        MI(X1, X2, ..., Xn) = sum(H(Xi)) - H(X1, X2, ..., Xn)
        """
        # Individual entropies
        individual_entropies = 0
        for var in variables:
            var_disc = self._discretize(var)
            counts = pd.Series(var_disc).value_counts()
            probs = counts / counts.sum()
            individual_entropies += entropy(probs, base=2)
        
        # Joint entropy
        discretized = [self._discretize(var) for var in variables]
        joint_df = pd.DataFrame({f'v{i}': discretized[i] for i in range(len(variables))})
        joint_df['joint'] = joint_df.astype(str).agg('_'.join, axis=1)
        
        counts = joint_df['joint'].value_counts()
        probs = counts / counts.sum()
        joint_entropy = entropy(probs, base=2)
        
        multi_info = individual_entropies - joint_entropy
        return multi_info
    
    def _o_information(self, *variables):
        """
        Calculate O-information for n variables.
        
        For triplets: O = -I(X;Y;Z)
        
        Where I(X;Y;Z) = interaction information = I(X;Y) + I(X;Z) + I(Y;Z) - I(X,Y;Z)
        
        Interpretation:
        I(X;Y;Z) > 0 → Synergy → O < 0
        I(X;Y;Z) < 0 → Redundancy → O > 0
        
        BUT we want intuitive signs, so return I directly:
        I > 0: Synergy (TFs cooperate)
        I < 0: Redundancy (TFs overlap)
        """
        n = len(variables)
        
        if n == 3:
            x, y, z = variables
            
            # Pairwise MIs
            mi_xy = mutual_info_score(self._discretize(x), self._discretize(y))
            mi_xz = mutual_info_score(self._discretize(x), self._discretize(z))
            mi_yz = mutual_info_score(self._discretize(y), self._discretize(z))
            
            # Joint MI I(X,Y;Z)
            x_disc = self._discretize(x)
            y_disc = self._discretize(y)
            z_disc = self._discretize(z)
            
            x_str = pd.Series(x_disc).astype(str)
            y_str = pd.Series(y_disc).astype(str)
            xy_joint = x_str + '_' + y_str
            xy_mapping = {v: i for i, v in enumerate(xy_joint.unique())}
            xy_numeric = xy_joint.map(xy_mapping).values
            
            mi_xyz = mutual_info_score(xy_numeric, z_disc)
            
            # Interaction information (synergy measure)
            interaction_info = mi_xy + mi_xz + mi_yz - mi_xyz
            
            tc = self._multi_information(*variables)
            
            # Return I directly (positive = synergy)
            return interaction_info, tc

    def _mutual_info_raw(self, x, y):
        """Calculate MI without using sklearn (for consistency)."""
        x_disc = self._discretize(x)
        y_disc = self._discretize(y)
        return mutual_info_score(x_disc, y_disc)
    
    def find_top_triplets(self, min_shared_targets=1, top_n=10):
        """
        Find top TF triplets (TF1, TF2, Target) by O-information.
        
        Returns:
            DataFrame with top triplets
        """
        print(f"\n=== Finding Top TF Triplets ===")
        
        # Get TF-target relationships
        tf_targets = {}
        for _, row in self.grn.iterrows():
            tf = row['TF']
            target = row['target']
            if tf not in tf_targets:
                tf_targets[tf] = set()
            tf_targets[tf].add(target)
        
        self.tf_targets = tf_targets
        
        # Find TF pairs with shared targets
        results = []
        tfs = [tf for tf in tf_targets.keys() if tf in self.expr.index]
        
        print(f"Evaluating {len(list(combinations(tfs, 2)))} TF pairs...")
        
        for tf1, tf2 in tqdm(combinations(tfs, 2), desc="Finding triplets", total=len(list(combinations(tfs, 2)))):
            shared_targets = tf_targets[tf1] & tf_targets[tf2]
            if len(shared_targets) < min_shared_targets:
                continue
            
            # Evaluate each shared target
            for target in shared_targets:
                if target not in self.expr.index:
                    continue
                
                tf1_expr = self.expr.loc[tf1].values
                tf2_expr = self.expr.loc[tf2].values
                target_expr = self.expr.loc[target].values
                
                try:
                    o_info, tc = self._o_information(tf1_expr, tf2_expr, target_expr)
                    
                    results.append({
                        'TF1': tf1,
                        'TF2': tf2,
                        'target': target,
                        'o_info': o_info,
                        'tc': tc
                    })
                except Exception as e:
                    continue
        
        triplets_df = pd.DataFrame(results).sort_values('o_info', ascending=False)
        
        print(f"Found {len(triplets_df)} valid triplets")
        print(f"\nTop {top_n} triplets:")
        print(triplets_df.head(top_n))
        
        return triplets_df.head(top_n)
    
    def greedy_expansion(self, initial_triplet, remaining_tfs, max_order=5):
        """
        Greedily add TFs to an initial triplet.
        
        Args:
            initial_triplet: Dict with keys TF1, TF2, target
            remaining_tfs: List of TFs to try adding
            max_order: Maximum combination order (including target)
        
        Returns:
            List of results for each expansion step
        """
        tf1 = initial_triplet['TF1']
        tf2 = initial_triplet['TF2']
        target = initial_triplet['target']
        
        current_tfs = [tf1, tf2]
        
        results = []
        
        # Baseline: order-3 (TF1, TF2, Target)
        exprs = [self.expr.loc[tf1].values, self.expr.loc[tf2].values, self.expr.loc[target].values]
        baseline, tc = self._o_information(*exprs)
        
        results.append({
            'order': 3,
            'tfs': current_tfs.copy(),
            'target': target,
            'o_info': baseline,
            'tc': tc
        })
        
        print(f"\n  Baseline ({tf1}, {tf2}, {target}): O-info = {baseline:.4f}")
        
        # Iteratively add TFs
        for order in range(4, max_order + 1):
            best_tf = None
            best_o_info = baseline
            best_tc = tc
            
            # Try adding each remaining TF
            for candidate_tf in remaining_tfs:
                if candidate_tf in current_tfs:
                    continue
                if candidate_tf not in self.expr.index:
                    continue
                
                # Check if candidate regulates the target
                if target not in self.tf_targets.get(candidate_tf, set()):
                    continue
                
                # Calculate O-info with candidate
                exprs = [self.expr.loc[tf].values for tf in current_tfs + [candidate_tf]] + [self.expr.loc[target].values]
                
                try:
                    candidate_o_info, candidate_tc = self._o_information(*exprs)
                except:
                    continue
                
                if candidate_o_info > best_o_info:
                    best_o_info = candidate_o_info
                    best_tf = candidate_tf
                    best_tc = candidate_tc
            
            # Stop if no improvement
            if best_tf is None or best_o_info <= baseline:
                print(f"  Order {order}: No improvement, stopping")
                break
            
            # Add best TF
            current_tfs.append(best_tf)
            baseline = best_o_info
            tc = best_tc
            
            results.append({
                'order': order,
                'tfs': current_tfs.copy(),
                'target': target,
                'o_info': best_o_info,
                'tc': best_tc
            })
            
            print(f"  Order {order}: Added {best_tf}, O-info = {best_o_info:.4f}")
        
        return results
    
    def find_optimal_combinations(self, top_triplets=10, max_order=6):
        """
        Main pipeline: find top triplets, then greedily expand each.
        
        Returns:
            DataFrame with all expansion results
        """
        # Step 1: Find top triplets
        triplets_df = self.find_top_triplets(top_n=top_triplets)
        
        if len(triplets_df) == 0:
            print("No valid triplets found!")
            return pd.DataFrame()
        
        # Step 2: Get all TFs
        all_tfs = list(self.tf_targets.keys())
        
        # Step 3: Expand each top triplet
        all_results = []
        
        print(f"\n=== Greedy Expansion ===")
        for idx, row in triplets_df.iterrows():
            initial_triplet = {
                'TF1': row['TF1'],
                'TF2': row['TF2'],
                'target': row['target']
            }
            
            remaining_tfs = [tf for tf in all_tfs if tf not in [row['TF1'], row['TF2']]]
            
            print(f"\nExpanding triplet {idx+1}/{len(triplets_df)}: ({row['TF1']}, {row['TF2']}, {row['target']})")
            
            expansion = self.greedy_expansion(initial_triplet, remaining_tfs, max_order=max_order)
            
            for result in expansion:
                result['initial_triplet'] = f"{row['TF1']}+{row['TF2']}→{row['target']}"
                result['stage'] = self.stage
                all_results.append(result)
        
        results_df = pd.DataFrame(all_results)
        
        # Summary
        print(f"\n=== Summary ===")
        print(f"Total combinations found: {len(results_df)}")
        print(f"\nBest combinations by order:")
        for order in sorted(results_df['order'].unique()):
            best = results_df[results_df['order'] == order].sort_values('o_info', ascending=False).iloc[0]
            print(f"  Order {order}: {best['tfs']} → {best['target']} - O-info = {best['o_info']:.4f}")
        
        return results_df


# === USAGE ===
# e14_finder = GreedyTFCombinationFinder(
#     grn_df=e14_grn,
#     counts_file=paths['counts'],
#     metadata_file=paths['metadata'],
#     deseq_file=paths['deseq_output'],
#     stage='E14',
#     n_bins=3
# )
# 
# e14_combinations = e14_finder.find_optimal_combinations(top_triplets=10, max_order=6)
# e14_combinations.to_csv('e14_optimal_tf_combinations.csv', index=False)

In [38]:

# === USAGE ===
e14_finder = GreedyTFCombinationFinder(
    grn_df=e14_grn_v2,
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    stage='E14',
    n_bins=3
)

e14_combinations = e14_finder.find_optimal_combinations(top_triplets=10, max_order=5)
e14_combinations.to_csv('/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/e14_optimal_tf_combination_v2s.csv', index=False)

# E18
e18_finder = GreedyTFCombinationFinder(
    grn_df=e18_grn_v2,
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    stage='E18',
    n_bins=3
)

e18_combinations = e18_finder.find_optimal_combinations(top_triplets=10, max_order=5)
e18_combinations.to_csv('/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/e18_optimal_tf_combinations_v2.csv', index=False)

Loaded expression for E14: 28656 genes, 10 samples

=== Finding Top TF Triplets ===
Evaluating 190 TF pairs...


Finding triplets: 100%|██████████| 190/190 [00:09<00:00, 20.92it/s]


Found 313 valid triplets

Top 10 triplets:
        TF1    TF2   target    o_info        tc
248   Sall4   Rorc   Tcf7l1  2.109840  3.043856
25   Tcf7l1  Sall4     Rorc  2.109840  3.043856
27   Tcf7l1  Sall4    Nr4a2  2.109840  3.043856
245   Sall4   Rorc    Nr4a2  2.109840  3.043856
48   Tcf7l1   Rorc    Nr4a2  2.109840  3.043856
23   Tcf7l1  Sall4     Ebf2  1.727932  2.492879
30   Tcf7l1  Sall4    Stat4  1.727932  2.492879
28   Tcf7l1  Sall4  Neurog1  1.727932  2.492879
246   Sall4   Rorc     Lef1  1.693952  2.443856
49   Tcf7l1   Rorc     Lef1  1.693952  2.443856

=== Greedy Expansion ===

Expanding triplet 249/10: (Sall4, Rorc, Tcf7l1)

  Baseline (Sall4, Rorc, Tcf7l1): O-info = 2.1098
  Order 4: No improvement, stopping

Expanding triplet 26/10: (Tcf7l1, Sall4, Rorc)

  Baseline (Tcf7l1, Sall4, Rorc): O-info = 2.1098
  Order 4: No improvement, stopping

Expanding triplet 28/10: (Tcf7l1, Sall4, Nr4a2)

  Baseline (Tcf7l1, Sall4, Nr4a2): O-info = 2.1098
  Order 4: No improvement, stop

Finding triplets: 100%|██████████| 630/630 [00:07<00:00, 85.57it/s] 


Found 253 valid triplets

Top 10 triplets:
       TF1     TF2 target    o_info        tc
244  Foxc1  Zfp366  Sox17  1.952866  2.817390
241  Foxc1   Sox18  Sox17  1.952866  2.817390
251  Sox18  Zfp366  Sox17  1.952866  2.817390
235   Hopx   Prrx2    Id3  1.952866  2.817390
14    Rora   Foxo1  Ppara  1.761912  2.541901
156   Etv4    Hopx    Id3  1.727932  2.768367
158   Etv4   Prrx2    Id3  1.727932  2.768367
252  Sox18  Zfp366   Fli1  1.570957  2.266412
243  Foxc1   Sox18   Fli1  1.570957  2.266412
245  Foxc1  Zfp366   Fli1  1.570957  2.266412

=== Greedy Expansion ===

Expanding triplet 245/10: (Foxc1, Zfp366, Sox17)

  Baseline (Foxc1, Zfp366, Sox17): O-info = 1.9529
  Order 4: No improvement, stopping

Expanding triplet 242/10: (Foxc1, Sox18, Sox17)

  Baseline (Foxc1, Sox18, Sox17): O-info = 1.9529
  Order 4: No improvement, stopping

Expanding triplet 252/10: (Sox18, Zfp366, Sox17)

  Baseline (Sox18, Zfp366, Sox17): O-info = 1.9529
  Order 4: No improvement, stopping

Expanding tr

In [39]:
e14_combinations

,order,tfs,target,o_info,tc,initial_triplet,stage
0,3,"[Sall4, Rorc]",Tcf7l1,2.109840,3.043856,Sall4+Rorc→Tcf7l1,E14
1,3,"[Tcf7l1, Sall4]",Rorc,2.109840,3.043856,Tcf7l1+Sall4→Rorc,E14
2,3,"[Tcf7l1, Sall4]",Nr4a2,2.109840,3.043856,Tcf7l1+Sall4→Nr4a2,E14
3,3,"[Sall4, Rorc]",Nr4a2,2.109840,3.043856,Sall4+Rorc→Nr4a2,E14
4,3,"[Tcf7l1, Rorc]",Nr4a2,2.109840,3.043856,Tcf7l1+Rorc→Nr4a2,E14
5,3,"[Tcf7l1, Sall4]",Ebf2,1.727932,2.492879,Tcf7l1+Sall4→Ebf2,E14
6,3,"[Tcf7l1, Sall4]",Stat4,1.727932,2.492879,Tcf7l1+Sall4→Stat4,E14
7,3,"[Tcf7l1, Sall4]",Neurog1,1.727932,2.492879,Tcf7l1+Sall4→Neurog1,E14
8,3,"[Sall4, Rorc]",Lef1,1.693952,2.443856,Sall4+Rorc→Lef1,E14
9,3,"[Tcf7l1, Rorc]",Lef1,1.693952,2.443856,Tcf7l1+Rorc→Lef1,E14


In [40]:
e18_combinations

,order,tfs,target,o_info,tc,initial_triplet,stage
0,3,"[Foxc1, Zfp366]",Sox17,1.952866,2.817390,Foxc1+Zfp366→Sox17,E18
1,3,"[Foxc1, Sox18]",Sox17,1.952866,2.817390,Foxc1+Sox18→Sox17,E18
2,3,"[Sox18, Zfp366]",Sox17,1.952866,2.817390,Sox18+Zfp366→Sox17,E18
3,3,"[Hopx, Prrx2]",Id3,1.952866,2.817390,Hopx+Prrx2→Id3,E18
4,3,"[Rora, Foxo1]",Ppara,1.761912,2.541901,Rora+Foxo1→Ppara,E18
5,3,"[Etv4, Hopx]",Id3,1.727932,2.768367,Etv4+Hopx→Id3,E18
6,3,"[Etv4, Prrx2]",Id3,1.727932,2.768367,Etv4+Prrx2→Id3,E18
7,3,"[Sox18, Zfp366]",Fli1,1.570957,2.266412,Sox18+Zfp366→Fli1,E18
8,3,"[Foxc1, Sox18]",Fli1,1.570957,2.266412,Foxc1+Sox18→Fli1,E18
9,3,"[Foxc1, Zfp366]",Fli1,1.570957,2.266412,Foxc1+Zfp366→Fli1,E18
